In [1]:
import pandas as pd
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import Metadata
from sdv.evaluation.single_table import evaluate_quality, get_column_plot
import os

In [2]:
# Constants
SYNTHETIC_SAMPLES = 10000

In [3]:
# Load the dataset
try:
    df = pd.read_csv("../../../Datasets/csv/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '.../../../Datasets/csv/spoofing-merged-gps-only.csv' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [4]:
# Data Preprocessing

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

if df['label'].dtype == object or df['label'].dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['label'])

In [5]:
metadata_file = "spoofing_metadata.json"
if os.path.exists(metadata_file):
    # Load existing metadata
    metadata = Metadata.load_from_json(metadata_file)
    print(f"Loaded metadata from {metadata_file}")
else:
    # Create and save new metadata
    metadata = Metadata()
    metadata.detect_table_from_dataframe(
        data=df,
        table_name='spoofing_data'
    )
    metadata.save_to_json(metadata_file)
    print(f"Created and saved new metadata to {metadata_file}")

In [6]:
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(df)

In [7]:
# Generate synthetic data and Save to CSV
synthetic_data = synthesizer.sample(num_rows=SYNTHETIC_SAMPLES)
print("Synthetic data preview:")
print(synthetic_data.head())
synthetic_data.to_csv("../../../Datasets/csv/spoofing-GaussianCopula-synthetic.csv", index=False)

Synthetic data preview:
   timestamp      q_0_      q_1_      q_2_      q_3_  delta_q_reset_0_  \
0   12109522 -0.268656  0.065012 -0.301161  0.714431          0.999995   
1    7780423 -0.236021  0.072450 -0.358598  0.819382          0.999995   
2   12327738 -0.314836  0.106469 -0.363921  0.653136          0.999995   
3    3410190 -0.377960 -0.020852  0.138246  0.945541          0.999995   
4    7728441 -0.206225  0.030248 -0.325573  0.899271          0.999995   

   delta_q_reset_1_  delta_q_reset_2_  delta_q_reset_3_  quat_reset_counter  \
0     -4.610000e-10      1.580000e-09          0.003237                   3   
1     -4.610000e-10      1.580000e-09          0.003237                   3   
2     -4.610000e-10      1.580000e-09          0.003237                   3   
3     -4.610000e-10      1.580000e-09          0.003237                   3   
4     -4.610000e-10      1.580000e-09          0.003237                   3   

   ...  v_z_valid  xy_reset_counter  z_reset_counter  vx

In [8]:
quality_report = evaluate_quality(
    real_data=df,
    synthetic_data=synthetic_data,
    metadata=metadata
)

# Show column distribution shapes comparison
quality_report.get_visualization('Column Shapes')

# Show column pair trends comparison
quality_report.get_visualization('Column Pair Trends')

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 84/84 [00:14<00:00,  5.82it/s]|
Column Shapes Score: 76.79%

(2/2) Evaluating Column Pair Trends: |██████████| 3486/3486 [00:11<00:00, 316.26it/s]|
Column Pair Trends Score: 80.27%

Overall Score (Average): 78.53%



In [9]:
fig = get_column_plot(
    real_data=df,
    synthetic_data=synthetic_data,
    column_name='lat_y',
    metadata=metadata
)

fig.show()

In [10]:
fig = get_column_plot(
    real_data=df,
    synthetic_data=synthetic_data,
    column_name='label',
    metadata=metadata
)

fig.show()

In [11]:
fig = get_column_plot(
    real_data=df,
    synthetic_data=synthetic_data,
    column_name='lat_y',
    metadata=metadata
)

fig.show()

In [12]:
fig = get_column_plot(
    real_data=df,
    synthetic_data=synthetic_data,
    column_name='label',
    metadata=metadata
)

fig.show()